<div style="background-color:#f5f3ff; border-radius:8px; padding:12px; text-align:center;">

# **PRACTICA 2: Juego de Tronos · VISUALIZACIÓN DE DATOS**

</div>

*Visualización de Juego de Tronos*

---

**Grupo:** G-7312  
**Número de pareja:** 01  
**Miembros:**  
- Leire Bernárdez Vázquez  
- Carmen Reiné Rueda


---

### **CONFIGURACIONES PREVIAS**

<div style="background-color:#f5f3ff; color:#6a0dad; padding:10px; border-radius:5px;">
Importaciones
</div>

In [1]:
import numpy as np
import pandas as pd
import json
import os
import glob
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
import networkx as nx       
import geopandas as gpd      
from folium.plugins import MarkerCluster
from collections import Counter

In [2]:
# === CARGA DE LOS DATOS ===

# Rutas base
ruta_data = r"C:\Users\carme\OneDrive - UAM\TERCERO\PRIMER CUATRI\VD\PRACTICAS\VD_G7312_P02\game_of_thrones\data"
ruta_map = r"C:\Users\carme\OneDrive - UAM\TERCERO\PRIMER CUATRI\VD\PRACTICAS\VD_G7312_P02\GOT-Map\GOT-Map"

# --- Carga de archivos JSON principales ---
with open(os.path.join(ruta_data, "characters.json"), "r", encoding="utf-8") as f:
    characters = json.load(f)

with open(os.path.join(ruta_data, "episodes.json"), "r", encoding="utf-8") as f:
    episodes = json.load(f)

with open(os.path.join(ruta_data, "locations.json"), "r", encoding="utf-8") as f:
    locations = json.load(f)

# --- Carga de archivos GeoJSON (mapas) ---
land = gpd.read_file(os.path.join(ruta_map, "land.geojson"))
places = gpd.read_file(os.path.join(ruta_map, "places.geojson"))

---


### **EJERCICIOS**



### <span >**Parte 1: Visualización de grafos con NetworkX**

En esta primera parte trabajaremos con las relaciones existentes entre los personajes de la serie *Game of Thrones* utilizando la librería **NetworkX**.  
El objetivo es construir el grafo de relaciones, obtener métricas y justificar las decisiones de diseño adoptadas (tipo de grafo, relaciones representadas, tamaño y color de nodos, y layout empleado).




<div style="background-color:#f5f3ff; color:#6a0dad; padding:10px; border-radius:5px;">
Apartado 1: Construcción del grafo de relaciones
</div>



A partir del fichero `characters.json`, construiremos un grafo donde los nodos representen personajes y las aristas indiquen algún tipo de relación entre ellos (parentesco, conflicto, etc.).


Antes de construir el grafo, realizamos una exploración básica del fichero para conocer cuántos personajes contiene, si hay registros sin nombre o duplicados, y qué información incluye. También miraremos todas las claves presentes en el dataset para distinguir cuáles son atributos descriptivos (nombre, actor, casa, etc.) y cuáles representan **relaciones entre personajes**.


In [10]:
data_chars = characters["characters"]
print(f"Personajes totales (sin limpieza): {len(data_chars)}")

# ¿Hay personajes SIN nombre?
sin_nombre = [i for i,c in enumerate(data_chars) if not c.get("characterName")]
print("\nSin nombre:", len(sin_nombre))

# ¿Hay nombres duplicados?
contador = Counter([c["characterName"] for c in data_chars if c.get("characterName")])
duplicados = [n for n,c in contador.items() if c>1]
print("\nDuplicados:", len(duplicados))
if duplicados:
    print(duplicados[:])  
    
# Todas las claves posibles en el JSON
todas_las_claves = sorted({k for c in data_chars for k in c.keys()})
print("\nTotal de claves en el JSON:", len(todas_las_claves))
print(todas_las_claves)


Personajes totales (sin limpieza): 389

Sin nombre: 0

Duplicados: 15
['Goldcloak', 'Handmaid', 'High Septon', 'Lannister Captain', 'Little Bird', 'Musician #1', 'Musician #2', 'Musician #3', "Night's Watch Officer", "Night's Watchman", "Night's Watchman #2", 'Red Priestess', 'Septon', 'Stark Guard', 'White Walker']

Total de claves en el JSON: 25
['abducted', 'abductedBy', 'actorLink', 'actorName', 'actors', 'allies', 'characterImageFull', 'characterImageThumb', 'characterLink', 'characterName', 'guardedBy', 'guardianOf', 'houseName', 'killed', 'killedBy', 'kingsguard', 'marriedEngaged', 'nickname', 'parentOf', 'parents', 'royal', 'servedBy', 'serves', 'sibling', 'siblings']


**1. Explora los tipos de relaciones disponibles en el fichero y selecciona al menos tres ipos diferentes para incluir en el grafo.**

A partir de la inspección previa, hemos seleccionado únicamente aquellas claves que representan vínculos entre personajes (por ejemplo: parents, parentOf, siblings, killed, killedBy, marriedEngaged, etc.).  
El resto de atributos (como characterName, actorName, houseName, etc.) se mantendrán solo si aportan información adicional en el grafo.

In [ ]:
# Claves que consideramos "de relación" según el JSON real
relaciones_validas = {
    "parents", "parentOf", "siblings",
    "killed", "killedBy",
    "marriedEngaged",
    "servedBy", "serves",
    "guardianOf", "guardedBy",
    "allies", "abducted", "abductedBy"
}

# Las que son realmente relaciones
relaciones_personajes = sorted({k for k in todas_las_claves if k in relaciones_validas})
print("\nSolo las claves que son relaciones válidas:", len(relaciones_personajes))
print("Tipos de relaciones presentes:", relaciones_personajes)




Solo las claves que son relaciones válidas: 13
Tipos de relaciones presentes: ['abducted', 'abductedBy', 'allies', 'guardedBy', 'guardianOf', 'killed', 'killedBy', 'marriedEngaged', 'parentOf', 'parents', 'servedBy', 'serves', 'siblings']


**2. Decide qué relaciones deben representarse como dirigidas y cuáles como no 
dirigidas**

In [ ]:
# === FRECUENCIA DE CADA RELACIÓN ===
conteo = Counter()

for c in data_chars:
    for k in relaciones_personajes:
        if isinstance(c.get(k), list):
            conteo[k] += len(c[k])

print("Frecuencia total (número de relaciones registradas):\n")
for k, v in conteo.items():
    print(f"{k:15s}: {v}")


Frecuencia total (número de relaciones registradas):

killedBy       : 206
parents        : 81
siblings       : 134
killed         : 222
marriedEngaged : 55
parentOf       : 78
servedBy       : 10
serves         : 22
guardedBy      : 12
guardianOf     : 15
allies         : 8
abductedBy     : 1
abducted       : 1
